# Langfuse Observability — Callback-Based LLM Tracing
## Traces in Production — Langfuse Callbacks for LangGraph
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/77-langfuse-observability/langfuse_observability_workbook.ipynb)

Instruments a LangGraph pipeline with a callback-based trace handler that mirrors the Langfuse interface. Runs fully in Colab without Langfuse credentials; shows how to swap in the real handler for production.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Observability layers: LLM calls, nodes, full runs; Langfuse vs LangSmith |
| 2 | **SimpleTraceHandler** — BaseCallbackHandler capturing on_llm_start / on_llm_end |
| 3 | **LangGraph + Callbacks** — Pass handler via callbacks= on ChatOpenAI |
| 4 | **Trace Inspection** — Print captured events; show trace_id per task |
| 5 | **Langfuse Swap-In** — One-line change to use real CallbackHandler in production |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — Observability Concepts

### Why observability in production?

In development you watch terminal output. In production, LLM calls happen inside HTTP requests — you can't watch them directly. Observability tools capture traces automatically so you can:

- Debug why a response was bad
- Audit what prompt was sent
- Track cost per request (token usage × model price)
- Spot regressions after a prompt change

### The callback protocol

LangChain uses a **callback handler** pattern: any object subclassing `BaseCallbackHandler` receives lifecycle events from LLM calls, chains, and tools. Langfuse, LangSmith, and other vendors ship their own handler — you just pass it in `callbacks=[handler]`.

### Trace → Span → Observation hierarchy

```
Trace (one user request)
  └── Span: graph invocation
        ├── Observation: LLM call in node A  (prompt, completion, tokens, latency)
        └── Observation: LLM call in node B
```

A **trace ID** threads through all spans so you can reconstruct the full request later.

### Langfuse vs LangSmith

| | Langfuse | LangSmith |
|--|---------|-----------|
| **Self-host** | Yes (Docker) | No |
| **OSS** | Yes | No |
| **LangGraph native** | Via callback | Via env var |
| **Cost tracking** | Yes | Yes |
| **Datasets / eval** | Yes | Yes |

> **Prerequisite:** Familiar with LangGraph nodes and LangSmith tracing (example 73).

In [ ]:
import uuid
from typing import TypedDict
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# BaseCallbackHandler mirrors the Langfuse CallbackHandler interface
# Swap in the real one with: from langfuse.callback import CallbackHandler
class SimpleTraceHandler(BaseCallbackHandler):
    def __init__(self, trace_id: str):
        super().__init__()
        self.trace_id  = trace_id
        self.events: list[dict] = []
        self.tokens_in  = 0
        self.tokens_out = 0

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.events.append({"event": "llm_start", "prompt_preview": prompts[0][:60]})

    def on_llm_end(self, response, **kwargs):
        gen = response.generations[0][0]
        text = gen.text if hasattr(gen, 'text') else str(gen)
        # capture token usage if the model returns it
        usage = getattr(response, "llm_output", {}) or {}
        tok   = usage.get("token_usage", {})
        self.tokens_in  += tok.get("prompt_tokens", 0)
        self.tokens_out += tok.get("completion_tokens", 0)
        self.events.append({"event": "llm_end", "output_preview": text[:60],
                            "tokens_in": self.tokens_in, "tokens_out": self.tokens_out})

    def summary(self):
        return {"trace_id": self.trace_id[:8], "events": len(self.events),
                "tokens_in": self.tokens_in, "tokens_out": self.tokens_out}

# demo: single LLM call with our trace handler
handler = SimpleTraceHandler(str(uuid.uuid4()))
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, callbacks=[handler])
resp = llm.invoke([HumanMessage(content="What is machine learning in one sentence?")])
print("Response:", resp.content[:100])
print("Trace summary:", handler.summary())
print("Events:")
for e in handler.events:
    print(" ", e)

## Part 2 — Multi-Node Traces with Correlation ID

When your graph has multiple LLM nodes, each call fires its own events. Without a shared correlation ID, you can't tell which events belong to the same user request.

The pattern:
1. Generate one `trace_id` (UUID) per user request
2. Store it in graph state
3. Pass the same handler instance to every LLM in the graph
4. All events on that handler share the same `trace_id`

In production Langfuse, this becomes the `trace_id` field on the `CallbackHandler`, automatically grouping all observations under one trace in the UI.

### Cost estimation

```
cost_usd = (tokens_in * price_per_input_token) + (tokens_out * price_per_output_token)
```

For `gpt-4o-mini` (as of early 2025): `$0.15 / 1M input`, `$0.60 / 1M output`.

In [ ]:
# Multi-node graph: think_node → answer_node, both traced under one trace_id
class QAState(TypedDict):
    question: str
    thinking: str
    answer:   str
    trace_id: str

GPT4O_MINI_PRICE = {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000}

def make_traced_graph(handler: SimpleTraceHandler):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, callbacks=[handler])

    def think_node(state: QAState) -> dict:
        r = llm.invoke([HumanMessage(content=f"Think briefly about: {state['question']}")])
        return {"thinking": r.content.strip()}

    def answer_node(state: QAState) -> dict:
        r = llm.invoke([HumanMessage(content=f"Answer concisely: {state['question']}")])
        return {"answer": r.content.strip()}

    g = StateGraph(QAState)
    g.add_node("think_node",  think_node)
    g.add_node("answer_node", answer_node)
    g.add_edge(START, "think_node")
    g.add_edge("think_node", "answer_node")
    g.add_edge("answer_node", END)
    return g.compile()

QUESTIONS = [
    "What is a neural network?",
    "What is gradient descent?",
    "What is backpropagation?",
]

all_summaries = []
for q in QUESTIONS:
    tid     = str(uuid.uuid4())
    handler = SimpleTraceHandler(tid)
    app     = make_traced_graph(handler)
    result  = app.invoke({"question": q, "thinking": "", "answer": "", "trace_id": tid})

    s   = handler.summary()
    est = (s["tokens_in"] * GPT4O_MINI_PRICE["input"] +
           s["tokens_out"] * GPT4O_MINI_PRICE["output"])
    all_summaries.append({**s, "cost_usd": est, "question": q})

    print(f"Q: {q}")
    print(f"  Answer: {result['answer'][:80]}")
    print(f"  Trace:  {s['trace_id']}  events={s['events']}  "
          f"tokens={s['tokens_in']}in/{s['tokens_out']}out  cost=${est:.6f}")
    print()

total_cost = sum(s["cost_usd"] for s in all_summaries)
print(f"Total estimated cost: ${total_cost:.6f}")

## Part 3 — Session Correlation and Tags

In a real app, a single user session may involve multiple graph invocations. You need two IDs:

- **`trace_id`** — unique per invocation (one LLM call chain)
- **`session_id`** — unique per user session (groups all traces from that session)

Additionally, **tags** and **metadata** let you slice traces in the UI:

```python
# Langfuse production usage
handler = CallbackHandler(
    public_key="...",
    secret_key="...",
    trace_id=str(uuid.uuid4()),
    session_id="user-session-abc123",
    tags=["production", "v2-prompt"],
    metadata={"user_id": "user_123", "feature_flag": "new_prompt"},
)
```

In our simulation below, we attach the same fields to the handler and print a session-level summary.

In [ ]:
class SessionTraceHandler(SimpleTraceHandler):
    def __init__(self, trace_id: str, session_id: str,
                 tags: list[str] | None = None, metadata: dict | None = None):
        super().__init__(trace_id)
        self.session_id = session_id
        self.tags       = tags or []
        self.metadata   = metadata or {}

    def summary(self):
        base = super().summary()
        return {**base, "session_id": self.session_id[:8],
                "tags": self.tags, "metadata": self.metadata}

# simulate two users, each with 2 invocations in one session
SESSION_A = str(uuid.uuid4())
SESSION_B = str(uuid.uuid4())

scenarios = [
    ("What is overfitting?",    SESSION_A, ["prod", "topic:ml"],   {"user": "alice"}),
    ("What is regularization?", SESSION_A, ["prod", "topic:ml"],   {"user": "alice"}),
    ("Explain transformers.",    SESSION_B, ["prod", "topic:nlp"],  {"user": "bob"}),
    ("What is attention?",       SESSION_B, ["prod", "topic:nlp"],  {"user": "bob"}),
]

session_data: dict[str, list] = {}
for question, sid, tags, meta in scenarios:
    tid     = str(uuid.uuid4())
    handler = SessionTraceHandler(tid, sid, tags=tags, metadata=meta)
    app     = make_traced_graph(handler)
    result  = app.invoke({"question": question, "thinking": "", "answer": "", "trace_id": tid})
    s       = handler.summary()
    session_data.setdefault(s["session_id"], []).append(s)
    print(f"[{s['session_id']}][{s['trace_id']}] {question[:50]}")
    print(f"  tags={s['tags']}  user={s['metadata'].get('user')}")

print()
for sid, traces in session_data.items():
    total_tok = sum(t["tokens_in"] + t["tokens_out"] for t in traces)
    print(f"Session {sid}: {len(traces)} traces, {total_tok} total tokens")

## Part 4 — Switching to Real Langfuse

Everything above runs without credentials. Switching to production Langfuse requires only:

1. Sign up at `cloud.langfuse.com` (free tier available) — or self-host with Docker
2. Grab your `LANGCHAIN_PUBLIC_KEY` and `LANGCHAIN_SECRET_KEY`
3. Replace `SimpleTraceHandler` with `CallbackHandler` from `langfuse`

```python
# Production switch — rest of code is IDENTICAL
from langfuse.callback import CallbackHandler

handler = CallbackHandler(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host="https://cloud.langfuse.com",   # or your self-hosted URL
    trace_id=str(uuid.uuid4()),
    session_id=session_id,
    tags=["production"],
)
```

### What you see in the Langfuse UI

- **Traces view**: list of all invocations, filterable by tag/session/date
- **Trace detail**: each span (node) with prompt, completion, latency, tokens
- **Sessions**: all traces for one user grouped together
- **Cost dashboard**: token usage and estimated cost by model and time range
- **Scores**: attach evaluation scores to traces (from RAGAS, manual review, etc.)

> Langfuse also supports **datasets** — upload your golden set, attach scores to traces automatically.

In [ ]:
# Simulate what the Langfuse UI would show — a structured trace report
import datetime

def format_trace_report(handler: SessionTraceHandler, question: str, answer: str) -> dict:
    return {
        "trace_id":    handler.trace_id,
        "session_id":  handler.session_id,
        "timestamp":   datetime.datetime.utcnow().isoformat() + "Z",
        "tags":        handler.tags,
        "metadata":    handler.metadata,
        "question":    question,
        "answer":      answer[:120],
        "observations": handler.events,
        "total_tokens": handler.tokens_in + handler.tokens_out,
        "cost_usd":    round(
            handler.tokens_in  * GPT4O_MINI_PRICE["input"] +
            handler.tokens_out * GPT4O_MINI_PRICE["output"],
            8
        ),
    }

# run one traced invocation and print a full report
import json as _json
q   = "Explain the attention mechanism in transformers."
tid = str(uuid.uuid4())
sid = str(uuid.uuid4())
h   = SessionTraceHandler(tid, sid, tags=["demo", "topic:nlp"], metadata={"user": "demo_user"})
app = make_traced_graph(h)
res = app.invoke({"question": q, "thinking": "", "answer": "", "trace_id": tid})
report = format_trace_report(h, q, res["answer"])
print(_json.dumps(report, indent=2))

## Exercises

---

**Exercise 1 — Add a Tool Node**

Add a third node `tool_node` between `think_node` and `answer_node` that performs a keyword extraction (use the LLM to extract 3 keywords from the thinking step). Verify that the trace captures 3 LLM observations instead of 2.

---

**Exercise 2 — Cost Per Node**

Modify `SimpleTraceHandler` to track tokens separately for each LLM call (not cumulative). After a 2-node run, print the token count and estimated cost per node.

*Hint:* Each `on_llm_start` → `on_llm_end` pair is one observation. Use an index or a stack.

---

**Exercise 3 — Score Attachment**

After running the eval pipeline from example 76, attach the `cosine_similarity` score to the trace report by adding a `"scores"` field. In real Langfuse this would be `langfuse.score(trace_id=..., name="cosine_similarity", value=...)`.

---

**Exercise 4 — Self-Hosted Langfuse**

Read the Langfuse self-hosting docs and write the `docker-compose.yml` command to start a local Langfuse instance. What environment variables do you need to set in your `.env`?

(This is a research + setup exercise — nothing to run in the notebook.)

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 2 — Cost Per Node (per-observation token tracking)
class PerNodeTraceHandler(BaseCallbackHandler):
    def __init__(self, trace_id: str):
        super().__init__()
        self.trace_id    = trace_id
        self.observations: list[dict] = []
        self._current: dict | None = None

    def on_llm_start(self, serialized, prompts, **kwargs):
        self._current = {"prompt_preview": prompts[0][:60], "tokens_in": 0, "tokens_out": 0}

    def on_llm_end(self, response, **kwargs):
        gen   = response.generations[0][0]
        usage = (getattr(response, "llm_output", {}) or {}).get("token_usage", {})
        obs   = {
            **(self._current or {}),
            "output_preview": (gen.text if hasattr(gen, 'text') else str(gen))[:60],
            "tokens_in":  usage.get("prompt_tokens", 0),
            "tokens_out": usage.get("completion_tokens", 0),
        }
        obs["cost_usd"] = (obs["tokens_in"]  * GPT4O_MINI_PRICE["input"] +
                           obs["tokens_out"] * GPT4O_MINI_PRICE["output"])
        self.observations.append(obs)
        self._current = None

h   = PerNodeTraceHandler(str(uuid.uuid4()))
app = make_traced_graph(h)
app.invoke({"question": "What is dropout in neural networks?",
            "thinking": "", "answer": "", "trace_id": h.trace_id})

print(f"Observations ({len(h.observations)}):")
for i, obs in enumerate(h.observations):
    print(f"  Obs {i+1}: {obs['tokens_in']}in / {obs['tokens_out']}out  cost=${obs['cost_usd']:.7f}")
    print(f"    Prompt preview: {obs['prompt_preview'][:60]}")

In [ ]:
# Answer 3 — Score Attachment
import numpy as np
from langchain_openai import OpenAIEmbeddings

embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

def cosine_sim(a: str, b: str) -> float:
    av = np.array(embed_model.embed_query(a))
    bv = np.array(embed_model.embed_query(b))
    return float(np.dot(av, bv) / (np.linalg.norm(av) * np.linalg.norm(bv)))

# run a traced eval with score attachment
question = "What is backpropagation?"
expected = "Backpropagation is an algorithm for computing gradients in neural networks."
tid = str(uuid.uuid4())
sid = str(uuid.uuid4())
h   = SessionTraceHandler(tid, sid, tags=["eval"], metadata={"eval_type": "cosine"})
app = make_traced_graph(h)
res = app.invoke({"question": question, "thinking": "", "answer": "", "trace_id": tid})

sim    = cosine_sim(expected, res["answer"])
report = format_trace_report(h, question, res["answer"])
report["scores"] = [{"name": "cosine_similarity", "value": round(sim, 4)}]

import json as _json
print(_json.dumps(report, indent=2))
# In real Langfuse:
# langfuse.score(trace_id=tid, name="cosine_similarity", value=sim)

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*